# pdf_vlm — Document QA experiment (Colab)

## What this experiment measures

For Hyundai WIA report packs at **5 / 10 / 20 / 50 / 100 pages**:

| Axis | Variants |
|---|---|
| OCR | PP-StructureV3 (**tables ON** on Colab) + PDF text enrich |
| RAG generation | **text-only** vs **multimodal** (page images) |
| Retrieval | **page-level** vs **hierarchical** |
| Metric | ANLS / EM / F1 + recall@k (needs Gemma GGUF for real answers) |

**Runtime:** GPU (T4+) recommended.

> Do **not** clone into `/content/pdf_vlm` — that folder name shadows the Python package.

## 0. Clone + install (package + OCR + llama.cpp)

In [ ]:
import sys, shutil
from pathlib import Path

REPO_URL = "https://github.com/mAn-He/pdf_vlm.git"
ROOT = Path("/content/pdf_vlm_repo")

# Remove shadowed clone path if present
shadow = Path("/content/pdf_vlm")
if shadow.exists() and shadow.resolve() != ROOT.resolve():
    shutil.rmtree(shadow, ignore_errors=True)

if not (ROOT / "pyproject.toml").exists():
    !git clone --depth 1 {REPO_URL} {ROOT}
else:
    print("Repo present:", ROOT)

%cd {ROOT}
for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))
print("cwd:", Path.cwd())

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path("/content/pdf_vlm_repo").resolve()
assert (ROOT / "src/pdf_vlm/utils/io.py").exists()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])
# index + OCR (tables) + viz
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[index,ocr,viz]"])

def try_install_llama():
    for url in [
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
        "https://abetlen.github.io/llama-cpp-python/whl/cu122",
        None,
    ]:
        cmd = [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"]
        if url:
            cmd += ["--extra-index-url", url]
        print("Trying llama-cpp:", url or "default")
        if subprocess.run(cmd).returncode == 0:
            return True
    return False

print("llama-cpp:", try_install_llama())

for k in list(sys.modules):
    if k == "pdf_vlm" or k.startswith("pdf_vlm."):
        del sys.modules[k]
sys.path.insert(0, str(ROOT / "src"))

import pdf_vlm
from pdf_vlm.utils.io import project_root
from pdf_vlm.ocr.paddle_structure import paddle_available
print("pdf_vlm:", pdf_vlm.__file__)
print("paddle:", paddle_available())

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Download Gemma GGUF (required for real QA answers)

1. Accept: https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf  
2. Colab secret `HF_TOKEN` or paste token  
3. Set `DOWNLOAD_GGUF = True` below

If GGUF is missing, the harness can still measure **retrieval**, but **ANLS/QA quality will be empty** (dry-run).

In [ ]:
from pathlib import Path
import os

DOWNLOAD_GGUF = True  # set False only if you intentionally skip generation

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")

if DOWNLOAD_GGUF and not (gguf.exists() and mmproj.exists()):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF token: ")
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
    !{sys.executable} scripts/download_models.py --with-mmproj

print("gguf:", gguf.exists(), gguf)
print("mmproj:", mmproj.exists(), mmproj)
HAS_GGUF = gguf.exists() and mmproj.exists()
print("HAS_GGUF:", HAS_GGUF)

## 2. Build length packs 5/10/20/50/100 (optional if already in repo)

Repo already ships truncated PDFs under `data/custom/{5,10,20,50,100}/`.  
If you uploaded full `QA_report_HW.pdf` to the repo root, you can rebuild packs here.

In [ ]:
from pathlib import Path
import sys

BUCKETS = "5,10,20,50,100"
src = Path("QA_report_HW.pdf")

if src.exists():
    !{sys.executable} scripts/prepare_hw_report_dataset.py --pdf {src} --buckets {BUCKETS}
else:
    print("No QA_report_HW.pdf in repo root — using existing data/custom packs.")

for b in [5, 10, 20, 50, 100]:
    d = Path(f"data/custom/{b}")
    pdfs = list(d.glob("*.pdf")) if d.exists() else []
    print(f"bucket={b}: pdfs={[p.name for p in pdfs]} q={(d/'questions.json').exists()}")

## 3. OCR (PP-StructureV3, tables ON) + build retrieval indexes

Uses `configs/ocr/pp_structure_v3_colab.yaml` (`use_table_recognition: true`).  
This is the step that actually pulls OCR/table text used by RAG.

In [ ]:
import sys

BUCKETS = "5,10,20,50,100"
# Start smaller if Colab RAM is tight: BUCKETS = "5,10,20"

!{sys.executable} scripts/colab_prepare_custom.py \
  --buckets {BUCKETS} \
  --no-stub \
  --enrich-pdf-text \
  --hash-embedder \
  --ocr-config ocr/pp_structure_v3_colab.yaml \
  --force

from pdf_vlm.utils.io import load_json, resolve_path
prep = load_json(resolve_path("data/custom/colab_prepared.json"))
print(prep)
assert prep.get("items"), "No docs prepared — check data/custom manifests/PDFs"

## 4. Run eval matrix (RAG variants × page lengths)

Cells: **text/multimodal × page/hierarchical × custom_{5,10,20,50,100}**  
If `HAS_GGUF` is False → forced dry-run (retrieval only, ANLS≈0).

In [ ]:
import subprocess, sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
mmproj = Path("models/mmproj-model-f16-4B.gguf")
HAS_GGUF = gguf.exists() and mmproj.exists()

cmd = [
    sys.executable, "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
    "--datasets", "custom_5,custom_10,custom_20,custom_50,custom_100",
    "--pipelines", "text,multimodal",
    "--retrievals", "page,hierarchical",
    "--top-k", "3",
    "--device", "cuda",
]
if not HAS_GGUF:
    cmd.append("--dry-run")
    print("WARNING: no GGUF → dry-run (retrieval only). Re-run section 1.")
else:
    print("GGUF found → full QA generation")

print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime, reverse=True)
print("latest:", runs[0] if runs else None)
if runs:
    for name in ["summary.json", "report.md", "aggregates.json"]:
        p = runs[0] / name
        if p.exists():
            print("====", name, "====")
            txt = p.read_text(encoding="utf-8")
            print(txt[:5000])
            break

## 5. Inference practicality bench (needs GGUF)

This cell intentionally skips if weights are missing — it is **not** the QA harness.

In [ ]:
import sys
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
if gguf.exists():
    !{sys.executable} scripts/bench_gemma_inference.py --repeats 2
else:
    print("Skip bench: models/gemma-3-4b-it-q4_0.gguf missing.")
    print("Fix: set DOWNLOAD_GGUF=True in section 1 and re-run that cell.")